### Step2: ingest Detroit parcels (assessed values plus geometry) into raw schema."

In [1]:
import os, pyproj, glob

candidates = glob.glob(r"C:\Users\cmray\anaconda3\envs\detroit-property\Library\share\proj\proj.db")
print("conda proj.db found:", candidates)

os.environ["PROJ_LIB"] = r"C:\Users\cmray\anaconda3\envs\detroit-property\Library\share\proj"
pyproj.datadir.set_data_dir(os.environ["PROJ_LIB"])

print("now using:", pyproj.datadir.get_data_dir())
print(pyproj.CRS.from_epsg(4326).name)

conda proj.db found: ['C:\\Users\\cmray\\anaconda3\\envs\\detroit-property\\Library\\share\\proj\\proj.db']
now using: C:\Users\cmray\anaconda3\envs\detroit-property\Library\share\proj
WGS 84


c:\Users\cmray\anaconda3\envs\detroit-property\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [2]:
from sqlalchemy import create_engine, text, URL
from getpass import getpass

# Prompts you to type your postgres password — it is NOT stored in the notebook.
password = getpass("Postgres password: ")

# Note the port: 5433, your non-standard one.
url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=password,
    host="localhost",
    port=5433,
    database="detroit_property",
)
engine = create_engine(url)

with engine.connect() as conn:
    print("Connected!")
    print(conn.execute(text("SELECT version();")).scalar())
    print(conn.execute(text("SELECT postgis_full_version();")).scalar())

Connected!
PostgreSQL 16.2, compiled by Visual C++ build 1937, 64-bit
POSTGIS="3.4.1 3.4.1" [EXTENSION] PGSQL="160" GEOS="3.12.1-CAPI-1.18.1" PROJ="8.2.1 NETWORK_ENABLED=OFF URL_ENDPOINT=https://cdn.proj.org USER_WRITABLE_DIRECTORY=C:\WINDOWS\ServiceProfiles\NetworkService\AppData\Local/proj DATABASE_PATH=C:\Program Files\PostgreSQL\16\share\contrib\postgis-3.4\proj\proj.db" LIBXML="2.9.14" LIBJSON="0.12" LIBPROTOBUF="1.2.1" WAGYU="0.5.0 (Internal)"


In [3]:
import requests
import pandas as pd

# Paste your GeoService URL here (the part ending in /FeatureServer/0).
# If yours ends in just /FeatureServer, add /0 for the first layer.
LAYER_URL = "https://services2.arcgis.com/qvkbeam7Wirps6zC/arcgis/rest/services/parcel_file_current/FeatureServer/0"
query_url = f"{LAYER_URL}/query"

params = {
    "where": "1=1",          # ArcGIS requires a filter; 1=1 means "everything"
    "outFields": "*",         # all columns
    "returnGeometry": "false",# skip geometry for the smoke test — we'll pull shapes tomorrow
    "f": "json",              # ask for ArcGIS JSON back
    "resultRecordCount": 5,   # just 5 rows for the smoke test
}

r = requests.get(query_url, params=params, timeout=60)
r.raise_for_status()
data = r.json()

features = data["features"]
print(f"Got {len(features)} rows")

# Each feature is {"attributes": {...}} — pull the attributes into a DataFrame
df = pd.DataFrame([f["attributes"] for f in features])
df

Got 5 rows


,object_id,parcel_id,address,zip_code,taxpayer_1,taxpayer_2,taxpayer_address,taxpayer_city,taxpayer_state,taxpayer_zip_code,...,subdivision,local_historic_district,neighborhood,council_district,street_number,street_prefix,street_name,ObjectId,Shape__Area,Shape__Length
0,243,02000184.,712 CASS,48226,"DETROIT CLUB HOLDINGS, LLC",,712 CASS AVE,DETROIT,MI,48226,...,,None,Downtown,6,712,,CASS,1,1541.785156,164.003882
1,24808,18007432.,1302 CRAWFORD,48209,"NAVA, JOSE",,1228 CASGRAIN,DETROIT,MI,48209,...,,None,Central Southwest,6,1302,,CRAWFORD,3,619.585938,125.262340
2,245,02000155.,404 W CONGRESS,48226,623 CASS AVE. LLC,,146-18 LIBERTY AVE,JAMIACA,NY,11435,...,,None,Downtown,6,404,W,CONGRESS,4,2354.699219,196.556745
3,24809,18007150.,1573 LIVERNOIS,48209,"REYES-CISNEROS, EDGAR",,1573 LIVERNOIS AVE,DETROIT,MI,48209-2051,...,,None,Central Southwest,6,1573,,LIVERNOIS,5,690.312500,136.680155
4,246,02000185-6,300 W FORT,48226,FREE PRESS HOLDINGS LLC,,315 LAKELAND AVENUE,GROSSE POINTE,MI,48230,...,,None,Downtown,6,300,W,FORT,6,3130.398438,224.033806


In [4]:
# Show every column name, not the truncated view
for col in df.columns:
    print(col)

object_id
parcel_id
address
zip_code
taxpayer_1
taxpayer_2
taxpayer_address
taxpayer_city
taxpayer_state
taxpayer_zip_code
property_class
property_class_description
property_class_previous
use_code
use_code_description
zoning_district
year_built
building_style
num_buildings
total_floor_area
tax_status
tax_status_description
tax_status_previous
amt_assessed_value
amt_assessed_value_previous
amt_taxable_value
amt_taxable_value_previous
pct_pre_claimed
nez_district
ecf_neighborhood
is_improved
related_parcel_id
sale_date
amt_sale_price
total_square_footage
total_acreage
frontage
depth
parcel_modified_date
legal_description
ward
landmap
subdivision
local_historic_district
neighborhood
council_district
street_number
street_prefix
street_name
ObjectId
Shape__Area
Shape__Length


In [5]:
import requests, geopandas as gpd

LAYER_URL = "https://services2.arcgis.com/qvkbeam7Wirps6zC/arcgis/rest/services/parcel_file_current/FeatureServer/0"

# Layer metadata — what the server will let us do
meta = requests.get(LAYER_URL, params={"f": "json"}, timeout=60).json()
print("maxRecordCount:", meta.get("maxRecordCount"))
print("geometryType:  ", meta.get("geometryType"))
print("spatialRef:    ", meta.get("extent", {}).get("spatialReference"))

# Two real features WITH geometry, as GeoJSON
r = requests.get(f"{LAYER_URL}/query", params={
    "where": "1=1", "outFields": "OBJECTID,parcel_id,amt_assessed_value,neighborhood",
    "returnGeometry": "true", "outSR": 4326, "f": "geojson", "resultRecordCount": 2,
}, timeout=60)
r.raise_for_status()
gdf = gpd.GeoDataFrame.from_features(r.json()["features"], crs="EPSG:4326")
print(gdf.geometry.geom_type.tolist())
print(gdf.crs)
gdf

maxRecordCount: 1000
geometryType:   esriGeometryPolygon
spatialRef:     {'wkid': 102100, 'latestWkid': 3857}
['Polygon', 'Polygon']
EPSG:4326


,geometry,ObjectId,parcel_id,amt_assessed_value,neighborhood
0,"POLYGON ((-83.05094 42.32975, -83.05072 42.329...",1,02000184.,1825900,Downtown
1,"POLYGON ((-83.10873 42.31104, -83.10872 42.311...",3,18007432.,1400,Central Southwest


In [6]:
import pyproj, os
print("pyproj:", pyproj.__version__)
print("data dir:", pyproj.datadir.get_data_dir())
print("exists:", os.path.exists(os.path.join(pyproj.datadir.get_data_dir(), "proj.db")))
print("PROJ_LIB env:", os.environ.get("PROJ_LIB"))
print("PROJ_DATA env:", os.environ.get("PROJ_DATA"))

pyproj: 3.7.2
data dir: C:\Users\cmray\anaconda3\envs\detroit-property\Library\share\proj
exists: True
PROJ_LIB env: C:\Users\cmray\anaconda3\envs\detroit-property\Library\share\proj
PROJ_DATA env: None
